In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ---------------------------------------------------------------------------
# Ausgabe-Verzeichnis (relativ zu jupyter/Hypothesis_testing.ipynb)
# ---------------------------------------------------------------------------
ASSETS_DIR = Path("../assets").resolve()
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, stem):
    """Schreibt PNG nach assets/."""
    fig.savefig(ASSETS_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")

# ---------------------------------------------------------------------------
# Globale Stil-Einstellungen
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
})

def fmt_p(p):
    """p-Wert in deutscher Notation, mit Schwelle für extrem kleine Werte."""
    if p < 1e-10:
        return "p < 10\u207B\u00B9\u2070"
    if p < 0.001:
        return "p < 0,001"
    return f"p = {p:.4f}".rstrip("0").rstrip(".").replace(".", ",")

de_pct = FuncFormatter(lambda x, _: f"{x:.2f}".replace(".", ",") + " %")
de_pp  = FuncFormatter(lambda x, _: f"{x:+.1f}".replace(".", ",") + " pp")
de_eur = FuncFormatter(lambda x, _: f"{x:+,.0f} €".replace(",", "."))


# ===========================================================================
# Plot 1 — Depletion Rate mit 95-%-Wilson-CI je Szenario × Strategie
# ===========================================================================
dep_rate = {
    "Standard": [
        ("Buy & Hold",  0.00, 0.00, 0.04),
        ("MSM",         0.00, 0.00, 0.04),
        ("HMM",         0.00, 0.00, 0.04),
        ("LSTM",        0.02, 0.01, 0.07),
        ("Transformer", 0.01, 0.00, 0.06),
    ],
    "Aggressive": [
        ("Buy & Hold",  4.58, 4.19, 5.01),
        ("MSM",         2.25, 1.98, 2.56),
        ("HMM",         0.53, 0.41, 0.69),
        ("LSTM",        3.73, 3.38, 4.12),
        ("Transformer", 4.06, 3.69, 4.46),
    ],
    "Low_Capital": [
        ("Buy & Hold",  0.62, 0.48, 0.79),
        ("MSM",         0.05, 0.02, 0.12),
        ("HMM",         0.00, 0.00, 0.04),
        ("LSTM",        0.27, 0.19, 0.39),
        ("Transformer", 0.38, 0.28, 0.52),
    ],
}

fig, axes = plt.subplots(3, 1, figsize=(7, 7), constrained_layout=True)

for ax, (scen, rows) in zip(axes, dep_rate.items()):
    strats = [r[0] for r in rows]
    pt = np.array([r[1] for r in rows])
    lo = np.array([r[2] for r in rows])
    hi = np.array([r[3] for r in rows])
    y  = np.arange(len(strats))[::-1]

    bh_pt = pt[0]
    ax.axvline(bh_pt, ls="--", lw=0.8, color="tab:grey", zorder=1,
               label="Buy & Hold-Referenz" if ax is axes[0] else None)

    ax.errorbar(pt, y,
                xerr=[pt - lo, hi - pt],
                fmt="o", color="black", ecolor="black",
                markersize=4.5, elinewidth=1, capsize=3, zorder=2)

    ax.set_yticks(y)
    ax.set_yticklabels(strats)
    ax.set_title(f"Szenario: {scen}", loc="left")
    ax.set_xlabel("Depletion Rate")
    ax.xaxis.set_major_formatter(de_pct)
    ax.grid(axis="x", lw=0.3, alpha=0.5)

    pad = 0.08 * max(hi.max(), 0.05)
    ax.set_xlim(-pad, hi.max() + pad)

axes[0].legend(loc="lower right", fontsize=8, frameon=False)
save(fig, "mcs_depletion_rate_forest")
plt.show()


# ===========================================================================
# Plot 2 — H1: Δ Median MaxDD (Modell vs. Buy & Hold), gepaarter Wilcoxon
# ===========================================================================
h1 = [
    ("MSM",         -6.93, 1.0000),
    ("HMM",         -5.99, 1.0000),
    ("LSTM",        +0.81, 0.0635),
    ("Transformer", -2.51, 1.0000),
]

fig, ax = plt.subplots(figsize=(7, 2.8), constrained_layout=True)

models = [r[0] for r in h1]
deltas = np.array([r[1] for r in h1])
pvals  = np.array([r[2] for r in h1])
sig    = pvals < 0.05
y      = np.arange(len(models))[::-1]

ax.axvline(0, ls="--", lw=0.8, color="tab:grey", label="kein Unterschied")
ax.scatter(deltas[~sig], y[~sig], s=55, facecolor="white",
           edgecolor="black", linewidth=1.2, zorder=3,
           label="nicht signifikant (p ≥ 0,05)")
ax.scatter(deltas[sig], y[sig], s=55, color="black", zorder=3,
           label="signifikant (p < 0,05)")

for d, yi, p in zip(deltas, y, pvals):
    offset = 10 if d >= 0 else -10
    ha     = "left" if d >= 0 else "right"
    ax.annotate(fmt_p(p), xy=(d, yi), xytext=(offset, 0),
                textcoords="offset points", va="center", ha=ha, fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels(models)
ax.set_xlabel("Δ Median MaxDD vs. Buy & Hold")
ax.xaxis.set_major_formatter(de_pp)
ax.grid(axis="x", lw=0.3, alpha=0.5)

xmin, xmax = deltas.min(), deltas.max()
span = xmax - xmin
ax.set_xlim(xmin - 0.35 * span, xmax + 0.35 * span)
ax.legend(loc="lower right", fontsize=8, frameon=False)

save(fig, "mcs_h1_mdd_forest")
plt.show()


# ===========================================================================
# Plot 3 — H2: Δ Median Endkapital (Transformer vs. Vergleichsmodell)
# ===========================================================================
h2 = [
    ("Transformer vs. MSM",  +33_208, 2.47e-81),
    ("Transformer vs. HMM",  +31_656, 1.95e-97),
    ("Transformer vs. LSTM", -24_196, 1.0000),
]

fig, ax = plt.subplots(figsize=(7, 2.4), constrained_layout=True)

labels = [r[0] for r in h2]
deltas = np.array([r[1] for r in h2])
pvals  = np.array([r[2] for r in h2])
sig    = pvals < 0.05
y      = np.arange(len(labels))[::-1]

ax.axvline(0, ls="--", lw=0.8, color="tab:grey", label="kein Unterschied")
ax.scatter(deltas[~sig], y[~sig], s=55, facecolor="white",
           edgecolor="black", linewidth=1.2, zorder=3,
           label="nicht signifikant (p ≥ 0,05)")
ax.scatter(deltas[sig], y[sig], s=55, color="black", zorder=3,
           label="signifikant (p < 0,05)")

for d, yi, p in zip(deltas, y, pvals):
    offset = 10 if d >= 0 else -10
    ha     = "left" if d >= 0 else "right"
    ax.annotate(fmt_p(p), xy=(d, yi), xytext=(offset, 0),
                textcoords="offset points", va="center", ha=ha, fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("Δ Median Endkapital")
ax.xaxis.set_major_formatter(de_eur)
ax.grid(axis="x", lw=0.3, alpha=0.5)

xmin, xmax = deltas.min(), deltas.max()
span = xmax - xmin
ax.set_xlim(xmin - 0.30 * span, xmax + 0.30 * span)
ax.legend(loc="lower right", fontsize=8, frameon=False)

save(fig, "mcs_h2_endkapital_forest")
plt.show()